# Klasifikasi Prioritas Tiket IT Helpdesk
TF-IDF + Multinomial Naive Bayes. Unggah `dataset_tiket_ti.csv` ke sesi Colab, lalu **Runtime > Run all**.

In [40]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score)
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

SEED = 42
df = pd.read_csv("dataset_tiket_ti.csv")

In [41]:
# 1. Pra-pemrosesan teks
def bersihkan(t):
    t = t.lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


STOP = {"di", "yang", "dan", "ke", "dari", "untuk", "saya", "mohon",
        "tolong", "ada", "ini", "itu", "dengan", "pada", "sudah",
        "atau", "agar", "halo", "selamat", "siang", "tim", "ti",
        "bantuan"}


def tok(t):
    return [w for w in bersihkan(t).split() if w not in STOP]


df["bersih"] = df["deskripsi"].apply(bersihkan)

In [42]:
# 2. Pembagian data 80:20 berstrata
X_tr, X_te, y_tr, y_te = train_test_split(
    df["bersih"], df["prioritas"], test_size=0.2,
    stratify=df["prioritas"], random_state=SEED)

In [43]:
# 3. Pipeline: TF-IDF (unigram+bigram) -> classifier
def buat_model(clf, ngram=(1, 2)):
    tfidf = TfidfVectorizer(tokenizer=tok, token_pattern=None,
                            ngram_range=ngram, sublinear_tf=True)
    return Pipeline([("tfidf", tfidf), ("clf", clf)])


model = buat_model(MultinomialNB(alpha=0.1)).fit(X_tr, y_tr)
pred = model.predict(X_te)

In [44]:
# 4. Evaluasi hold-out
labels = ["Rendah", "Sedang", "Tinggi"]
print("Akurasi uji:", accuracy_score(y_te, pred))
print(classification_report(y_te, pred, digits=3))
print(confusion_matrix(y_te, pred, labels=labels))

Akurasi uji: 0.975
              precision    recall  f1-score   support

      Rendah      1.000     1.000     1.000        13
      Sedang      1.000     0.929     0.963        14
      Tinggi      0.929     1.000     0.963        13

    accuracy                          0.975        40
   macro avg      0.976     0.976     0.975        40
weighted avg      0.977     0.975     0.975        40

[[13  0  0]
 [ 0 13  1]
 [ 0  0 13]]


In [45]:
# 5. Validasi silang 5 lipatan
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
cv = cross_val_score(buat_model(MultinomialNB(alpha=0.1)),
                     df["bersih"], df["prioritas"], cv=skf)
print("CV:", cv.round(3), "rerata", cv.mean().round(3))

CV: [0.975 0.9   0.975 0.925 0.9  ] rerata 0.935


In [46]:
# 6. Perbandingan algoritme
for nama, clf in [("Naive Bayes", MultinomialNB(alpha=0.1)),
                  ("Logistic Regression",
                   LogisticRegression(max_iter=1000, C=10)),
                  ("Linear SVM", LinearSVC(C=1.0))]:
    s = cross_val_score(buat_model(clf), df["bersih"],
                        df["prioritas"], cv=skf)
    print(f"{nama}: {s.mean():.3f} +/- {s.std():.3f}")

Naive Bayes: 0.935 +/- 0.034
Logistic Regression: 0.935 +/- 0.030
Linear SVM: 0.940 +/- 0.025


In [47]:
# 7. Aturan keputusan: ambang kepercayaan + kata kunci keamanan
nb = model.named_steps["clf"]
KUNCI = ["peretasan", "diretas", "phishing", "ransomware", "bocor",
         "serangan", "malware", "mencurigakan"]


def putuskan(teks, ambang=0.70):
    p = model.predict_proba([bersihkan(teks)])[0]
    kelas, conf = nb.classes_[p.argmax()], float(p.max())
    if conf < ambang or any(k in teks.lower() for k in KUNCI):
        return kelas, conf, "TINJAU PETUGAS"
    return kelas, conf, "OTOMATIS"

In [48]:
# 8. Demonstrasi tiket baru
baru = [
    "server SIAKAD tidak bisa diakses semua mahasiswa saat pengisian KRS",
    "printer di ruang dosen macet kertasnya",
    "mohon bantuan reset password email kampus",
    "wifi lantai 3 putus dan ada indikasi serangan siber",
    "minta panduan penggunaan e-learning untuk mahasiswa baru",
    "laptop saya lambat sekali",
]
for t in baru:
    print(t, "->", putuskan(t))

server SIAKAD tidak bisa diakses semua mahasiswa saat pengisian KRS -> (np.str_('Tinggi'), 0.9879802385571482, 'OTOMATIS')
printer di ruang dosen macet kertasnya -> (np.str_('Sedang'), 0.9198801125838828, 'OTOMATIS')
mohon bantuan reset password email kampus -> (np.str_('Rendah'), 0.9130853915680727, 'OTOMATIS')
wifi lantai 3 putus dan ada indikasi serangan siber -> (np.str_('Sedang'), 0.9341596326936937, 'TINJAU PETUGAS')
minta panduan penggunaan e-learning untuk mahasiswa baru -> (np.str_('Rendah'), 0.9814341730393484, 'OTOMATIS')
laptop saya lambat sekali -> (np.str_('Sedang'), 0.951861888870418, 'OTOMATIS')
